# Stage 4 — With GAN, With Drift
### GPVS-Faults | Journal extension of ICSPCS 2024 & ITNAC 2026

Stage 4 = Stage 2's drift injection (`drift_injection.py`, unchanged) +
Stage 3's GAN augmentation (`gan_augmentation.py`, unchanged) + the same two
classifiers. This is the roadmap's **Cell 4 — the main research question**:
does GAN augmentation mitigate drift-induced damage?

**Pipeline ordering matters and is worth stating explicitly.** Drift
injection happens to the real data FIRST (identical to Stage 2), and the
GAN is trained on that already-drifted real training partition SECOND.
This is required, not just convenient: `wgans.py`'s `augment()` zero-fills
the Time column for synthetic rows, so there is no meaningful Time value to
compute a drift tau from at augmentation time. Injecting drift before the
GAN ever sees the data sidesteps that entirely for the baseline variant
(4a) — the GAN just learns and reproduces the already-drifted training
distribution, real Time not required.

## Two variants in this notebook

**Stage 4a — vanilla GAN + drift (the literal 2×2×2 Cell 4).** The GAN
learns the drifted TRAINING distribution (local tau roughly 0–0.625, per
Stage 2's per-class timeline) and generates more of the same. Tests whether
simply having more (synthetic, same-distribution) training data helps the
classifier generalize better to the drifted TEST distribution (tau roughly
0.8125–1.0) — a real question, but the GAN never explicitly sees anything
resembling test-time severity.

**Stage 4b — drift-aware synthetic augmentation ("domain adaptation").**
Takes 4a's synthetic rows and additionally projects them forward along the
same parametric drift ramp (`project_to_tau`), sampling tau from the TEST
partition's range instead of the range the GAN actually learned from. This
gives the classifier labeled synthetic examples resembling the TARGET
(test-time) distribution during training — using the fact that the exact
parametric drift mechanism is known, which is a much stronger assumption
than a generic augmentation method gets to make, but is exactly what this
study's controlled design licenses. **This is the more novel variant and
the one worth featuring as the "domain adaptation" contribution if it beats
4a by more than seed noise.**

**Recommended reporting**: run both. 4a is the necessary baseline cell for
the 2×2×2 table regardless of outcome; 4b is the interesting row if — and
only if — it beats 4a by a margin that survives `multiseed.paired_diff`,
given the ~4.7-pt LSTM-XGB seed noise already established in Stage 2.

> **Execution note:** as in Stage 3, every GPU-bound cell here is ready-to-run
> but not executed in this sandbox. The drift-injection and manipulation-check
> cells (Phase 3/4, unchanged from Stage 2) *were* executed for real, against
> the actual `base_splits.pkl`.


In [ ]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, "..")
from base_splits import load_splits, summarize_splits, save_splits, build_base_splits, load_dataframe
from drift_injection import (detrend_vdc_normal, inject_drift, calibrate_amplitude,
                              manipulation_check, DRIFT_FEATURES,
                              VDC_FLOOR_W_RAW, VDC_FLOOR_KS_RAW)
from gan_augmentation import gan_augment_splits, splits_train_to_array, project_to_tau, adaptation_only_augment, gan_augment_full_spectrum, gan_augment_stratified, build_stratified_multiseverity_pool, DEFAULT_TAU_BINS
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, paired_diff, DEFAULT_SEEDS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from scipy.stats import ks_2samp, wasserstein_distance
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters

splits_raw = load_splits(path="../base_splits.pkl")
print(summarize_splits(splits_raw))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = list(splits_raw[0].train.columns[1:-1])
print("feature cols:", cols)


[detrend check] label 0: KS=0.3265 (p=1.58e-25), Wasserstein=0.1457  [pre-detrend baseline: KS~=0.553, Wasserstein~=1.089]
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409
Saved base splits for 8 classes to /home/maddie/cd-study/Stages/stage4/base_splits.pkl


AttributeError: Can't get attribute 'ClassSplit' on <module '__main__'>

## Phase 3/4 — Drift Injection (unchanged from Stage 2)

Identical code and identical calibrated amplitudes to Stage 2 — the point
of Stage 4 is to add GAN augmentation on top of the *same* drift condition,
not a different one. Re-executed here rather than loaded from a pickle so
this notebook is self-contained.


In [2]:
splits_detrended, vdc_trend_info = detrend_vdc_normal(splits_raw, normal_label=0)
for lbl in range(1, 8):
    assert (splits_detrended[lbl].train["Vdc"].to_numpy()
            == splits_raw[lbl].train["Vdc"].to_numpy()).all()
print("Fault-class Vdc rows confirmed untouched by detrending.")
for tag, s in [("raw", splits_raw), ("detrended", splits_detrended)]:
    tr, te = s[0].train["Vdc"].to_numpy(), s[0].test["Vdc"].to_numpy()
    print(f"Normal Vdc train-vs-test [{tag:>10}]  "
          f"KS={ks_2samp(tr, te).statistic:.4f}  W={wasserstein_distance(tr, te):.4f}")


Fault-class Vdc rows confirmed untouched by detrending.
Normal Vdc train-vs-test [       raw]  KS=0.3807  W=0.1609
Normal Vdc train-vs-test [ detrended]  KS=0.1550  W=0.0938


In [3]:
scaler_ref = StandardScaler().fit(splits_raw[0].train[cols])
sigma = dict(zip(cols, scaler_ref.scale_))

TARGET_SEVERITY_X = 4.0
floor_z = VDC_FLOOR_W_RAW / sigma["Vdc"]
target_z = TARGET_SEVERITY_X * floor_z
print(f"target ({TARGET_SEVERITY_X}x floor, z-units): {target_z:.4f}\n")

amplitudes = {}
for feat in DRIFT_FEATURES:
    target_raw = target_z * sigma[feat]
    amp, w = calibrate_amplitude(splits_detrended, feat, target_raw, scope="all")
    amplitudes[feat] = amp
    print(f"  {feat:5s}  amplitude={amp:.5f}  achieved_mean_W={w:.5f}  (target={target_raw:.5f})")

splits_stage2, tau_lookup = inject_drift(splits_detrended, amplitudes,
                                         features=DRIFT_FEATURES, scope="all")
print("\nDrift injected -- splits_stage2 ready (identical to Stage 2's dataset).")


target (4.0x floor, z-units): 6.9384

  Ipv    amplitude=1.18171  achieved_mean_W=0.69683  (target=0.68754)
  Vpv    amplitude=3.40364  achieved_mean_W=2.02585  (target=2.02635)


  Iabc   amplitude=0.03516  achieved_mean_W=0.02146  (target=0.02163)

Drift injected -- splits_stage2 ready (identical to Stage 2's dataset).


## Stage 4a — GAN Augmentation on the Drifted Training Data

Same `gan_augment_splits` call as Stage 3, fed `splits_stage2` (drifted)
instead of `splits_raw`. The GAN learns whatever the drifted training
partition looks like — it has no knowledge that drift was injected, it
just sees a training distribution and imitates it.


In [4]:
GAN_SEED = 20260827          # same fixed seed as Stage 3, for consistency
GAN_AUGMENT_RATIO = 1.0

splits_stage4a, gans_4, gan_histories_4, n_real = gan_augment_splits(
    splits_stage2, seed=GAN_SEED, device=device, ratio=GAN_AUGMENT_RATIO)

print(f"Trained {len(gans_4)} per-class WGAN-GP generators on drifted training data.")
for lbl, hist in sorted(gan_histories_4.items()):
    print(f"  class {lbl}: final W_dist={hist['w_dist'][-1]:.4f}")


/home/maddie/miniconda3/envs/cd/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Trained 8 per-class WGAN-GP generators on drifted training data.
  class 0: final W_dist=0.3063
  class 1: final W_dist=0.2984
  class 2: final W_dist=0.2665
  class 3: final W_dist=0.2222
  class 4: final W_dist=0.2730
  class 5: final W_dist=0.3331
  class 6: final W_dist=0.2949
  class 7: final W_dist=0.2703


In [5]:
fidelity_rows = []
for lbl in sorted(splits_stage4a):
    real = splits_stage2[lbl].train
    aug = splits_stage4a[lbl].train
    synth = aug.iloc[len(real):]
    for feat in cols:
        ks = ks_2samp(real[feat], synth[feat]).statistic
        w = wasserstein_distance(real[feat], synth[feat])
        fidelity_rows.append({"class": lbl, "feature": feat, "ks": ks, "wasserstein": w})
fidelity = pd.DataFrame(fidelity_rows)
print("GAN fidelity (drifted real train vs. synthetic), mean across classes:")
print(fidelity.pivot_table(index="feature", values=["ks", "wasserstein"], aggfunc="mean").round(4))


GAN fidelity (drifted real train vs. synthetic), mean across classes:
             ks  wasserstein
feature                     
Iabc     0.0770       0.0005
If       0.0798       0.0095
Ipv      0.0629       0.0233
Vabc     0.0841       0.0057
Vdc      0.2255       0.1530
Vf       0.0800       0.0005
Vpv      0.0635       0.0783
ia       0.0546       0.0544
ib       0.0510       0.0446
ic       0.0494       0.0306
va       0.0542       5.9671
vb       0.0488       6.6473
vc       0.0508       5.2419


## Stage 4b — Drift-Aware Synthetic Augmentation

Take 4a's synthetic rows (which resemble the drifted TRAINING distribution,
local tau roughly 0–0.625) and additionally push them forward to the TEST
partition's tau range (~0.8125–1.0) using the same parametric ramp
(`value - amplitude * tau`) that generated the drift in the first place.
Same real rows, same total synthetic count, same GAN — the only difference
from 4a is where along the drift trajectory the synthetic rows sit.

`TEST_TAU_RANGE` below is the approximate tau span of the test partition
under uniform time sampling (300 of 1600 rows, i.e. the last 18.75%): each
class's test partition spans roughly tau∈[0.8125, 1.0]. This is an
approximation — computable exactly per class from `tau_lookup` above if
tighter precision is wanted.


In [6]:
TEST_TAU_RANGE = (0.8125, 1.0)
rng = np.random.default_rng(GAN_SEED)

splits_stage4b = {lbl: splits_stage4a[lbl] for lbl in splits_stage4a}  # shallow; train replaced below
import copy
splits_stage4b = copy.deepcopy(splits_stage4a)

for lbl in sorted(splits_stage4b):
    aug = splits_stage4b[lbl].train
    n_real_lbl = len(splits_stage2[lbl].train)
    real_part = aug.iloc[:n_real_lbl]
    synth_part = aug.iloc[n_real_lbl:].reset_index(drop=True)
    tau_sample = rng.uniform(TEST_TAU_RANGE[0], TEST_TAU_RANGE[1], size=len(synth_part))
    synth_projected = project_to_tau(synth_part, amplitudes, DRIFT_FEATURES, tau=tau_sample)
    splits_stage4b[lbl].train = pd.concat([real_part, synth_projected], axis=0).reset_index(drop=True)

print("splits_stage4b ready: same real rows + drift-aware-projected synthetic rows.")
print(f"Synthetic rows projected to tau ~ Uniform{TEST_TAU_RANGE} "
      f"(vs. 4a's implicit tau ~ training range).")


splits_stage4b ready: same real rows + drift-aware-projected synthetic rows.
Synthetic rows projected to tau ~ Uniform(0.8125, 1.0) (vs. 4a's implicit tau ~ training range).


## Stage 4c — Adaptation-Only (No GAN)

Decompose 4b into its two components and test the adaptation piece alone.
4b = **generalization** (WGAN-GP smooths/densifies the real training rows
into a continuous manifold) + **adaptation** (`project_to_tau` injects known
target-domain information). 4c removes the generalization component
entirely: bootstrap-resample the REAL training rows and project them with
the exact same formula and the exact same target tau range 4b uses — no
generator involved at all.

This is the missing cell needed to answer "is the hybrid actually earning
its cost, or is the projection step carrying the whole effect?" Two
possible outcomes, both informative:
  - **4c underperforms 4b**: the GAN's richer, smoothed coverage of the
    training manifold genuinely helps the projection step generalize
    better — the hybrid (generalization + adaptation) is a real, novel
    combination worth naming as such.
  - **4c matches or beats 4b**: the projection step was doing all the work;
    the GAN is unnecessary complexity, and the simpler, cheaper
    adaptation-only method is the one to report.


In [7]:
splits_stage4c = adaptation_only_augment(
    splits_stage2, cols=cols, features=DRIFT_FEATURES, amplitudes=amplitudes,
    ratio=GAN_AUGMENT_RATIO, tau_range=TEST_TAU_RANGE, seed=GAN_SEED)

print("splits_stage4c ready: real rows + resampled-and-projected real rows (no GAN).")
for lbl in sorted(splits_stage4c):
    print(f"  class {lbl}: train n = {len(splits_stage4c[lbl].train)}")


splits_stage4c ready: real rows + resampled-and-projected real rows (no GAN).
  class 0: train n = 2000
  class 1: train n = 2000
  class 2: train n = 2000
  class 3: train n = 2000
  class 4: train n = 2000
  class 5: train n = 2000
  class 6: train n = 2000
  class 7: train n = 2000


## Stage 4d — Row-Count Control (No Shift, No GAN)

The control 4c needs. 4c's "extra" 1000 rows are bootstrap-resampled real
rows with only the 3 drift features shifted — the other 10 features are
verbatim copies of real data. That means 4c isn't cleanly "adaptation vs.
generation": it also has zero synthesis noise on 10/13 features, by
construction, which is a second advantage entangled with the shift itself.

4d isolates this: identical bootstrap duplication, identical final row
count (2000/class), but **no shift at all** — plain oversampling with zero
target-domain information. If 4d already matches 4c, the tau-shift isn't
doing the work either, and 4c's edge over 4b would be pure sample-fidelity,
not drift-awareness. If 4c clearly beats 4d, the shift is earning its keep
independent of the row-count question.


In [8]:
splits_stage4d = adaptation_only_augment(
    splits_stage2, cols=cols, features=DRIFT_FEATURES, amplitudes=amplitudes,
    ratio=GAN_AUGMENT_RATIO, tau_range=TEST_TAU_RANGE, seed=GAN_SEED,
    apply_projection=False)   # <-- only difference from 4c: no shift applied

print("splits_stage4d ready: real rows + bootstrap-duplicated real rows, NO shift, NO GAN.")
for lbl in sorted(splits_stage4d):
    print(f"  class {lbl}: train n = {len(splits_stage4d[lbl].train)}")


splits_stage4d ready: real rows + bootstrap-duplicated real rows, NO shift, NO GAN.
  class 0: train n = 2000
  class 1: train n = 2000
  class 2: train n = 2000
  class 3: train n = 2000
  class 4: train n = 2000
  class 5: train n = 2000
  class 6: train n = 2000
  class 7: train n = 2000


## Stage 4f — Full-Spectrum Pool, Unconditional GAN

Tests the idea directly: instead of projecting the GAN's *output* to the
target tau range (4b), inject the tau-shift into the GAN's *training
input* across the full 0-1 spread, so the generator is exposed to
examples resembling the entire drift trajectory during its own training.

**Read before interpreting the result**: this GAN is unconditional — it is
never told which tau a given pool row came from, so it can only learn one
blended (marginal) distribution mixing low- and high-severity examples
together. It has no mechanism to selectively generate test-like samples on
demand the way `project_to_tau`'s deterministic, targeted shift does. This
is expected to underperform 4b/4c for that specific reason — not because
"training on more spread" is a bad idea in general, but because doing it
without conditioning the generator on tau throws away targeting precision.
A properly-engineered version of this idea needs a tau-conditioned
generator (CcGAN-style) — a real architecture change, scoped separately,
not implemented here. This cell tests the cheap, honest version of the
idea empirically rather than assuming the outcome.


In [9]:
splits_stage4f, gans_4f, histories_4f = gan_augment_full_spectrum(
    splits_stage2, cols=cols, features=DRIFT_FEATURES, amplitudes=amplitudes,
    seed=GAN_SEED, device=device, pool_multiplier=1.0, ratio=GAN_AUGMENT_RATIO,
    tau_range=(0.0, 1.0))

print("splits_stage4f ready: real rows + GAN-generated rows, GAN trained on a full-tau-spectrum pool.")
for lbl in sorted(splits_stage4f):
    print(f"  class {lbl}: train n = {len(splits_stage4f[lbl].train)}")


splits_stage4f ready: real rows + GAN-generated rows, GAN trained on a full-tau-spectrum pool.
  class 0: train n = 2000
  class 1: train n = 2000
  class 2: train n = 2000
  class 3: train n = 2000
  class 4: train n = 2000
  class 5: train n = 2000
  class 6: train n = 2000
  class 7: train n = 2000


## Stage 4g — Stratified Multi-Severity Pool (the proposed framework)

The framework this study is building toward: **one GAN trained on a pool
that spans the whole deployment lifetime** — genuinely undrifted data,
graded drift severities, and extrapolation beyond anything observed —
producing an enlarged training set intended to address sample scarcity
and concept drift together, rather than treating them as separate problems.

Pool composition, per class (equal 20% tiers by default):

| Tier | tau | Represents |
|---|---|---|
| `pristine` | 0.0 exactly | Pre-deployment baseline, no degradation |
| `early` | 0.0–0.300 | Early-life mild degradation |
| `observed` | 0.300–0.625 | The severity range real training data spans |
| `test_horizon` | 0.625–1.000 | The unseen near-future (val + test region) |
| `beyond` | 1.000–1.500 | Extrapolated degradation past anything observed |

**Absolute vs. additive tau — why this pool is built differently.**
4c/4d/4f build their pools from `splits_stage2`, where every row already
carries its own local drift, so their projection lands *on top of* an
existing shift and the resulting absolute severity is uncontrolled. 4g
builds from `splits_detrended` (PRE-injection) and applies an **absolute**
tau per tier, which is what makes "this tier is severity X" actually true.
That precision is the whole point of stratifying.

The real rows the synthetic data is appended to are still Stage 2's
drifted training rows — identical to every other Stage-4 variant — so the
row count stays at 2x real and the comparison stays fair. Only the GAN's
*training pool* differs.


In [10]:
splits_stage4g, gans_4g, histories_4g, pool_4g = gan_augment_stratified(
    splits_detrended,        # GAN pool source: PRE-drift data, absolute tau applied per tier
    splits_stage2,           # real rows appended to: same drifted rows as 4a-4f
    cols=cols, features=DRIFT_FEATURES, amplitudes=amplitudes,
    seed=GAN_SEED, device=device,
    tau_bins=DEFAULT_TAU_BINS, bin_fractions=None,   # None -> equal 20% tiers
    pool_multiplier=1.0, ratio=GAN_AUGMENT_RATIO)

print("\nsplits_stage4g ready: real drifted rows + GAN rows from the stratified lifetime pool.")
for lbl in sorted(splits_stage4g):
    print(f"  class {lbl}: train n = {len(splits_stage4g[lbl].train)}")


  pool composition (per class, n=1000): pristine[0.000-0.000]:200  early[0.000-0.300]:200  observed[0.300-0.625]:200  test_horizon[0.625-1.000]:200  beyond[1.000-1.500]:200



splits_stage4g ready: real drifted rows + GAN rows from the stratified lifetime pool.
  class 0: train n = 2000
  class 1: train n = 2000
  class 2: train n = 2000
  class 3: train n = 2000
  class 4: train n = 2000
  class 5: train n = 2000
  class 6: train n = 2000
  class 7: train n = 2000


### Pool sanity check

Confirm each tier actually sits at its intended severity before trusting
anything downstream — Vpv and Ipv should step down monotonically across
tiers, and the `pristine` tier should match the true undrifted data.


In [11]:
p = pool_4g[0].train
n_per = len(p) // len(DEFAULT_TAU_BINS)
print("Class 0 pool, mean by tier (should decrease monotonically):")
for i, (name, lo, hi) in enumerate(DEFAULT_TAU_BINS):
    seg = p.iloc[i*n_per:(i+1)*n_per]
    print(f"  {name:13s} tau[{lo:.3f},{hi:.3f}]  Vpv={seg['Vpv'].mean():8.4f}  Ipv={seg['Ipv'].mean():7.4f}")
print(f"\n  true undrifted Vpv mean = {splits_detrended[0].train['Vpv'].mean():.4f}"
      f"  (should match the pristine tier)")


Class 0 pool, mean by tier (should decrease monotonically):
  pristine      tau[0.000,0.000]  Vpv= 90.4800  Ipv= 2.3131
  early         tau[0.000,0.300]  Vpv= 89.9546  Ipv= 2.1109
  observed      tau[0.300,0.625]  Vpv= 88.9735  Ipv= 1.7679
  test_horizon  tau[0.625,1.000]  Vpv= 87.7498  Ipv= 1.3622
  beyond        tau[1.000,1.500]  Vpv= 86.2646  Ipv= 0.8364

  true undrifted Vpv mean = 90.5078  (should match the pristine tier)


In [12]:
scaler = StandardScaler()
scaler.fit(splits_raw[0].train[cols])   # frozen -- identical fit to every prior stage

def build_dct(splits_variant):
    dct = dict()
    for i in range(len(splits_variant)):
        dct[i] = dict()
        dct[i].update({
            "train": pd.DataFrame(scaler.transform(splits_variant[i].train[cols]), columns=cols,
                                  index=splits_variant[i].train.index).assign(Fault=i),
            "val": pd.DataFrame(scaler.transform(splits_variant[i].val[cols]), columns=cols,
                                index=splits_variant[i].val.index).assign(Fault=i),
            "test": pd.DataFrame(scaler.transform(splits_variant[i].test[cols]), columns=cols,
                                 index=splits_variant[i].test.index).assign(Fault=i),
        })
    return dct

dct_4a = build_dct(splits_stage4a)
dct_4b = build_dct(splits_stage4b)
dct_4c = build_dct(splits_stage4c)
dct_4d = build_dct(splits_stage4d)
dct_4f = build_dct(splits_stage4f)
dct_4g = build_dct(splits_stage4g)
print("dct_4a:", {i: len(dct_4a[i]["train"]) for i in dct_4a})
print("dct_4b:", {i: len(dct_4b[i]["train"]) for i in dct_4b})
print("dct_4c:", {i: len(dct_4c[i]["train"]) for i in dct_4c})
print("dct_4d:", {i: len(dct_4d[i]["train"]) for i in dct_4d})
print("dct_4f:", {i: len(dct_4f[i]["train"]) for i in dct_4f})
print("dct_4g:", {i: len(dct_4g[i]["train"]) for i in dct_4g})


dct_4a: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4b: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4c: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4d: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4f: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}
dct_4g: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000, 6: 2000, 7: 2000}


## Model Evaluation — All Four Variants

Unchanged model code. Run once each against `dct_4a`, `dct_4b`, `dct_4c`, `dct_4d`.


In [13]:
def to_tensors(df, cols):
    """Convert a (features + Fault) DataFrame into model-ready tensors.
    X: (N, 1, len(cols)) so the LSTM sees the len(cols) features as a
       length-len(cols) sequence with 1 channel each (matches LSTM_XGB's
       expected input shape).
    y: (N,) integer Fault labels.
    """
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y


def load_scenario(dct, cols):
    """Build combined 8-class train/val/test tensors directly from the
    in-memory `dct` dict (built in the Z-scale cell), instead of reading
    per-scenario CSVs off disk.

    dct is keyed by class label: dct[i]["train"/"val"/"test"] is a
    per-class DataFrame of z-scored features + a Fault column. We
    concatenate across classes to get the full multiclass split.
    """
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols),
            to_tensors(val_df,   cols),
            to_tensors(test_df,  cols))


def run_scenario(scenario_idx, dct, cols, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


### LSTM-XGB — Stage 4a (vanilla GAN + drift)

In [14]:
print(f"LSTM-XGB (4a) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4a_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4a, cols=cols, device=device)
lstm_xgb_4a_agg = aggregate_results(lstm_xgb_4a_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4a_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4a) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.7396  P=0.7234  R=0.7396  F1=0.6994  train_sec=92.3


  seed=1  acc=0.7063  P=0.8666  R=0.7063  F1=0.6666  train_sec=91.9


  seed=2  acc=0.7742  P=0.8635  R=0.7742  F1=0.7333  train_sec=91.2


  seed=3  acc=0.8367  P=0.7577  R=0.8367  F1=0.7889  train_sec=90.2


  seed=4  acc=0.8342  P=0.7663  R=0.8342  F1=0.7861  train_sec=86.5

mean +/- std:
  accuracy    0.7782 +/- 0.0575
  precision   0.7955 +/- 0.0655
  recall      0.7782 +/- 0.0575
  f1          0.7349 +/- 0.0535


### LSTM-XGB — Stage 4b (drift-aware / domain adaptation)

In [15]:
print(f"LSTM-XGB (4b) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4b_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4b, cols=cols, device=device)
lstm_xgb_4b_agg = aggregate_results(lstm_xgb_4b_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4b_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4b) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9567  P=0.9620  R=0.9567  F1=0.9536  train_sec=87.0


  seed=1  acc=0.9187  P=0.9461  R=0.9187  F1=0.9084  train_sec=86.9


  seed=2  acc=0.9983  P=0.9983  R=0.9983  F1=0.9983  train_sec=86.8


  seed=3  acc=0.9471  P=0.9628  R=0.9471  F1=0.9447  train_sec=87.1


  seed=4  acc=0.9350  P=0.9474  R=0.9350  F1=0.9337  train_sec=88.7

mean +/- std:
  accuracy    0.9512 +/- 0.0299
  precision   0.9633 +/- 0.0211
  recall      0.9512 +/- 0.0299
  f1          0.9478 +/- 0.0330


### LSTM-XGB — Stage 4c (adaptation-only, no GAN)

In [16]:
print(f"LSTM-XGB (4c) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4c_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4c, cols=cols, device=device)
lstm_xgb_4c_agg = aggregate_results(lstm_xgb_4c_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4c_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4c) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9571  P=0.9642  R=0.9571  F1=0.9566  train_sec=88.4


  seed=1  acc=0.9054  P=0.9288  R=0.9054  F1=0.9027  train_sec=88.9


  seed=2  acc=0.9663  P=0.9733  R=0.9663  F1=0.9656  train_sec=88.5


  seed=3  acc=0.9213  P=0.9416  R=0.9212  F1=0.9183  train_sec=88.6


  seed=4  acc=0.8879  P=0.9173  R=0.8879  F1=0.8821  train_sec=88.3

mean +/- std:
  accuracy    0.9276 +/- 0.0334
  precision   0.9450 +/- 0.0235
  recall      0.9276 +/- 0.0334
  f1          0.9251 +/- 0.0355


### LSTM-XGB — Stage 4d (row-count control, no shift, no GAN)

In [17]:
print(f"LSTM-XGB (4d) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4d_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4d, cols=cols, device=device)
lstm_xgb_4d_agg = aggregate_results(lstm_xgb_4d_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4d_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4d) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.6833  P=0.6052  R=0.6833  F1=0.6235  train_sec=88.8


  seed=1  acc=0.7937  P=0.8548  R=0.7937  F1=0.7667  train_sec=78.8


  seed=2  acc=0.6121  P=0.5348  R=0.6121  F1=0.5130  train_sec=74.8


  seed=3  acc=0.6837  P=0.6684  R=0.6838  F1=0.6386  train_sec=75.3


  seed=4  acc=0.7417  P=0.8635  R=0.7417  F1=0.7215  train_sec=75.9

mean +/- std:
  accuracy    0.7029 +/- 0.0685
  precision   0.7053 +/- 0.1482
  recall      0.7029 +/- 0.0685
  f1          0.6527 +/- 0.0978


In [18]:
# CNN-LSTM training pipeline -- byte-identical to stage1.ipynb. to_tensors()
# and load_scenario() are reused as-is from the previous cell.

def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device,
                          epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


### CNN-LSTM-v2 — Stage 4a (vanilla GAN + drift)

In [19]:
MODEL_CLS = CNN_LSTM_v2

print(f"CNN-LSTM-v2 (4a) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4a_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4a, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4a_agg = aggregate_results(cnn_lstm_4a_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4a_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4a) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.8746  P=0.8035  R=0.8746  F1=0.8297  train_sec=120.5


  seed=1  acc=0.8750  P=0.8119  R=0.8750  F1=0.8331  train_sec=116.5


  seed=2  acc=0.8742  P=0.8106  R=0.8742  F1=0.8322  train_sec=119.8


  seed=3  acc=0.8746  P=0.8108  R=0.8746  F1=0.8325  train_sec=113.5


  seed=4  acc=0.8750  P=0.8122  R=0.8750  F1=0.8332  train_sec=93.0

mean +/- std:
  accuracy    0.8747 +/- 0.0003
  precision   0.8098 +/- 0.0036
  recall      0.8747 +/- 0.0003
  f1          0.8321 +/- 0.0014


### CNN-LSTM-v2 — Stage 4b (drift-aware / domain adaptation)

In [20]:
print(f"CNN-LSTM-v2 (4b) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4b_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4b, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4b_agg = aggregate_results(cnn_lstm_4b_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4b_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4b) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9571  P=0.9681  R=0.9571  F1=0.9558  train_sec=113.2


  seed=1  acc=0.9979  P=0.9979  R=0.9979  F1=0.9979  train_sec=120.4


  seed=2  acc=0.9775  P=0.9809  R=0.9775  F1=0.9773  train_sec=120.7


  seed=3  acc=0.9992  P=0.9992  R=0.9992  F1=0.9992  train_sec=117.7


  seed=4  acc=0.9496  P=0.9641  R=0.9496  F1=0.9474  train_sec=116.5

mean +/- std:
  accuracy    0.9762 +/- 0.0228
  precision   0.9820 +/- 0.0163
  recall      0.9762 +/- 0.0228
  f1          0.9755 +/- 0.0237


### CNN-LSTM-v2 — Stage 4c (adaptation-only, no GAN)

In [21]:
print(f"CNN-LSTM-v2 (4c) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4c_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4c, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4c_agg = aggregate_results(cnn_lstm_4c_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4c_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4c) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9796  P=0.9824  R=0.9796  F1=0.9794  train_sec=120.0


  seed=1  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=116.6


  seed=2  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=119.5


  seed=3  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=120.3


  seed=4  acc=0.9879  P=0.9890  R=0.9879  F1=0.9879  train_sec=115.8

mean +/- std:
  accuracy    0.9933 +/- 0.0092
  precision   0.9941 +/- 0.0080
  recall      0.9933 +/- 0.0092
  f1          0.9933 +/- 0.0093


### CNN-LSTM-v2 — Stage 4d (row-count control, no shift, no GAN)

In [22]:
print(f"CNN-LSTM-v2 (4d) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4d_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4d, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4d_agg = aggregate_results(cnn_lstm_4d_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4d_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4d) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.8746  P=0.8047  R=0.8746  F1=0.8301  train_sec=120.2


  seed=1  acc=0.8750  P=0.8122  R=0.8750  F1=0.8332  train_sec=121.7


  seed=2  acc=0.8750  P=0.8122  R=0.8750  F1=0.8332  train_sec=118.8


  seed=3  acc=0.8742  P=0.8064  R=0.8742  F1=0.8306  train_sec=120.8


  seed=4  acc=0.8750  P=0.8116  R=0.8750  F1=0.8330  train_sec=119.9

mean +/- std:
  accuracy    0.8747 +/- 0.0004
  precision   0.8094 +/- 0.0036
  recall      0.8747 +/- 0.0004
  f1          0.8320 +/- 0.0015


### LSTM-XGB — Stage 4f (full-spectrum pool, unconditional GAN)

In [23]:
print(f"LSTM-XGB (4f) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4f_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4f, cols=cols, device=device)
lstm_xgb_4f_agg = aggregate_results(lstm_xgb_4f_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4f_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4f) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9946  P=0.9947  R=0.9946  F1=0.9946  train_sec=60.2


  seed=1  acc=0.9750  P=0.9792  R=0.9750  F1=0.9749  train_sec=81.6


  seed=2  acc=0.9333  P=0.9556  R=0.9333  F1=0.9308  train_sec=81.7


  seed=3  acc=0.9712  P=0.9764  R=0.9712  F1=0.9709  train_sec=81.9


  seed=4  acc=0.8892  P=0.9261  R=0.8892  F1=0.8846  train_sec=82.2

mean +/- std:
  accuracy    0.9527 +/- 0.0419
  precision   0.9664 +/- 0.0265
  recall      0.9527 +/- 0.0419
  f1          0.9512 +/- 0.0438


### CNN-LSTM-v2 — Stage 4f (full-spectrum pool, unconditional GAN)

In [24]:
print(f"CNN-LSTM-v2 (4f) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4f_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4f, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4f_agg = aggregate_results(cnn_lstm_4f_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4f_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4f) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9983  P=0.9984  R=0.9983  F1=0.9983  train_sec=121.0


  seed=1  acc=0.8783  P=0.9380  R=0.8783  F1=0.8405  train_sec=119.4


  seed=2  acc=0.9900  P=0.9907  R=0.9900  F1=0.9900  train_sec=120.2


  seed=3  acc=0.9592  P=0.9688  R=0.9592  F1=0.9580  train_sec=120.7


  seed=4  acc=0.9408  P=0.9596  R=0.9408  F1=0.9373  train_sec=121.5

mean +/- std:
  accuracy    0.9533 +/- 0.0479
  precision   0.9711 +/- 0.0243
  recall      0.9533 +/- 0.0479
  f1          0.9448 +/- 0.0633


### LSTM-XGB — Stage 4g (stratified multi-severity pool)

In [25]:
print(f"LSTM-XGB (4g) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_4g_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct_4g, cols=cols, device=device)
lstm_xgb_4g_agg = aggregate_results(lstm_xgb_4g_results)
print("\nmean +/- std:")
for m, s in lstm_xgb_4g_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


LSTM-XGB (4g) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9158  P=0.9494  R=0.9158  F1=0.9052  train_sec=82.1


  seed=1  acc=0.9454  P=0.9543  R=0.9454  F1=0.9407  train_sec=81.8


  seed=2  acc=0.9950  P=0.9951  R=0.9950  F1=0.9950  train_sec=81.7


  seed=3  acc=0.9554  P=0.9604  R=0.9554  F1=0.9520  train_sec=81.3


  seed=4  acc=0.9113  P=0.9341  R=0.9112  F1=0.9094  train_sec=75.7

mean +/- std:
  accuracy    0.9446 +/- 0.0339
  precision   0.9587 +/- 0.0226
  recall      0.9446 +/- 0.0339
  f1          0.9405 +/- 0.0364


### CNN-LSTM-v2 — Stage 4g (stratified multi-severity pool)

In [26]:
print(f"CNN-LSTM-v2 (4g) across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_4g_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct_4g, cols=cols,
    model_cls=MODEL_CLS, device=device)
cnn_lstm_4g_agg = aggregate_results(cnn_lstm_4g_results)
print("\nmean +/- std:")
for m, s in cnn_lstm_4g_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}")


CNN-LSTM-v2 (4g) across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9663  P=0.9734  R=0.9663  F1=0.9656  train_sec=120.2


  seed=1  acc=0.9683  P=0.9745  R=0.9683  F1=0.9678  train_sec=119.1


  seed=2  acc=0.9079  P=0.9467  R=0.9079  F1=0.8934  train_sec=106.6


  seed=3  acc=0.9342  P=0.9569  R=0.9342  F1=0.9293  train_sec=108.6


  seed=4  acc=0.9050  P=0.9457  R=0.9050  F1=0.8889  train_sec=119.1

mean +/- std:
  accuracy    0.9363 +/- 0.0305
  precision   0.9595 +/- 0.0140
  recall      0.9363 +/- 0.0305
  f1          0.9290 +/- 0.0378


## Combined Summary — All Variants (4a / 4b / 4c / 4d / 4f / 4g, both models)

In [27]:
combined = summary_table({
    "LSTM-XGB (4a)": lstm_xgb_4a_agg, "LSTM-XGB (4b)": lstm_xgb_4b_agg,
    "LSTM-XGB (4c)": lstm_xgb_4c_agg, "LSTM-XGB (4d)": lstm_xgb_4d_agg,
    "LSTM-XGB (4f)": lstm_xgb_4f_agg, "LSTM-XGB (4g)": lstm_xgb_4g_agg,
    "CNN-LSTM-v2 (4a)": cnn_lstm_4a_agg, "CNN-LSTM-v2 (4b)": cnn_lstm_4b_agg,
    "CNN-LSTM-v2 (4c)": cnn_lstm_4c_agg, "CNN-LSTM-v2 (4d)": cnn_lstm_4d_agg,
    "CNN-LSTM-v2 (4f)": cnn_lstm_4f_agg, "CNN-LSTM-v2 (4g)": cnn_lstm_4g_agg,
})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    "LSTM-XGB 4a": per_class_accuracy(lstm_xgb_4a_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4b": per_class_accuracy(lstm_xgb_4b_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4c": per_class_accuracy(lstm_xgb_4c_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4d": per_class_accuracy(lstm_xgb_4d_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4f": per_class_accuracy(lstm_xgb_4f_agg["mean_confusion_rate"], class_names),
    "LSTM-XGB 4g": per_class_accuracy(lstm_xgb_4g_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4a": per_class_accuracy(cnn_lstm_4a_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4b": per_class_accuracy(cnn_lstm_4b_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4c": per_class_accuracy(cnn_lstm_4c_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4d": per_class_accuracy(cnn_lstm_4d_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4f": per_class_accuracy(cnn_lstm_4f_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM 4g": per_class_accuracy(cnn_lstm_4g_agg["mean_confusion_rate"], class_names),
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())
print("\n>>> Check F3 specifically -- it's the class most worth watching given Stage 2's finding. <<<")

import pickle
with open("stage4_seed_results.pkl", "wb") as f:
    pickle.dump({
        "lstm_xgb_4a": lstm_xgb_4a_agg, "lstm_xgb_4b": lstm_xgb_4b_agg,
        "lstm_xgb_4c": lstm_xgb_4c_agg, "lstm_xgb_4d": lstm_xgb_4d_agg,
        "lstm_xgb_4f": lstm_xgb_4f_agg, "lstm_xgb_4g": lstm_xgb_4g_agg,
        "cnn_lstm_v2_4a": cnn_lstm_4a_agg, "cnn_lstm_v2_4b": cnn_lstm_4b_agg,
        "cnn_lstm_v2_4c": cnn_lstm_4c_agg, "cnn_lstm_v2_4d": cnn_lstm_4d_agg,
        "cnn_lstm_v2_4f": cnn_lstm_4f_agg, "cnn_lstm_v2_4g": cnn_lstm_4g_agg,
    }, f)
print("\nSaved stage4_seed_results.pkl")


           model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB (4a)        5         0.7782        0.0575          0.7955         0.0655       0.7782      0.0575   0.7349  0.0535
   LSTM-XGB (4b)        5         0.9512        0.0299          0.9633         0.0211       0.9512      0.0299   0.9478  0.0330
   LSTM-XGB (4c)        5         0.9276        0.0334          0.9450         0.0235       0.9276      0.0334   0.9251  0.0355
   LSTM-XGB (4d)        5         0.7029        0.0685          0.7053         0.1482       0.7029      0.0685   0.6527  0.0978
   LSTM-XGB (4f)        5         0.9527        0.0419          0.9664         0.0265       0.9527      0.0419   0.9512  0.0438
   LSTM-XGB (4g)        5         0.9446        0.0339          0.9587         0.0226       0.9446      0.0339   0.9405  0.0364
CNN-LSTM-v2 (4a)        5         0.8747        0.0003          0.8098         0.0036       0.8747      

## Key Comparisons (paired by seed)

- **Cell 3 vs. Cell 4a**: does vanilla GAN augmentation mitigate drift damage at all?
- **Cell 2 vs. Cell 4a**: does drift erode the GAN's normal (no-drift) benefit?
- **Cell 4a vs. Cell 4b**: does drift-aware ("domain adaptation") projection
  beat vanilla augmentation, by more than seed noise?
- **Cell 2 vs. Cell 4d**: does merely duplicating real rows (no shift, no
  GAN) help at all? This should be the weakest condition of the four.
- **Cell 4d vs. Cell 4c — isolates the shift**: does the tau-projection add
  anything beyond plain oversampling at the same row count?
- **Cell 4c vs. Cell 4b — the hybrid question**: does adding the GAN's
  generalization step on top of the same projection meaningfully beat
  adaptation alone? If not significant, 4c is the method to report —
  simpler and cheaper, with no generator training at all.


In [28]:
import pickle
with open("../stage2/stage2_seed_results.pkl", "rb") as f:
    stage2 = pickle.load(f)
with open("../stage3/stage3_seed_results.pkl", "rb") as f:
    stage3 = pickle.load(f)

pairs = [
    ("Cell 3 (Stage 2, no GAN) vs Cell 4a (GAN, vanilla)",
     stage2, "lstm_xgb", lstm_xgb_4a_agg, "cnn_lstm_v2", cnn_lstm_4a_agg),
    ("Cell 2 (Stage 3, GAN no-drift) vs Cell 4a (GAN, drift)",
     stage3, "lstm_xgb", lstm_xgb_4a_agg, "cnn_lstm_v2", cnn_lstm_4a_agg),
]
for label, other_stage, key_a, agg_a_lstm, key_b, agg_a_cnn in pairs:
    print(f"\n=== {label} ===")
    for model_key, model_label, agg_new in [
        ("lstm_xgb", "LSTM-XGB", lstm_xgb_4a_agg),
        ("cnn_lstm_v2", "CNN-LSTM-v2", cnn_lstm_4a_agg),
    ]:
        base_results = other_stage[model_key]["raw_results"]
        new_results = agg_new["raw_results"]
        diff_df, diff_stats = paired_diff(new_results, base_results, metric="accuracy")
        print(f"  {model_label}: mean diff = {diff_stats['mean_diff']:+.4f}  "
              f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== Cell 4a (vanilla) vs Cell 4b (domain-adaptation) ===")
for model_label, results_4a, results_4b in [
    ("LSTM-XGB", lstm_xgb_4a_agg["raw_results"], lstm_xgb_4b_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4a_agg["raw_results"], cnn_lstm_4b_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4b, results_4a, metric="accuracy")
    print(f"  {model_label}: mean diff (4b - 4a) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== Cell 2 (Stage 2, no GAN) vs Cell 4c (adaptation-only, no GAN) ===")
for model_label, key, results_4c in [
    ("LSTM-XGB", "lstm_xgb", lstm_xgb_4c_agg["raw_results"]),
    ("CNN-LSTM-v2", "cnn_lstm_v2", cnn_lstm_4c_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4c, stage2[key]["raw_results"], metric="accuracy")
    print(f"  {model_label}: mean diff (4c - Stage2) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== Cell 2 (Stage 2, no GAN) vs Cell 4d (row-count control, no shift) ===")
for model_label, key, results_4d in [
    ("LSTM-XGB", "lstm_xgb", lstm_xgb_4d_agg["raw_results"]),
    ("CNN-LSTM-v2", "cnn_lstm_v2", cnn_lstm_4d_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4d, stage2[key]["raw_results"], metric="accuracy")
    print(f"  {model_label}: mean diff (4d - Stage2) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== THE SHIFT QUESTION: Cell 4d (oversample only) vs Cell 4c (oversample + shift) ===")
for model_label, results_4d, results_4c in [
    ("LSTM-XGB", lstm_xgb_4d_agg["raw_results"], lstm_xgb_4c_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4d_agg["raw_results"], cnn_lstm_4c_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4c, results_4d, metric="accuracy")
    sig = "SIGNIFICANT -- the tau-shift itself is doing real work" if diff_stats["p_value"] < 0.05 \
        else "NOT significant -- plain oversampling explains most/all of 4c's effect"
    print(f"  {model_label}: mean diff (4c - 4d) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")

print("\n=== THE HYBRID QUESTION: Cell 4c (adaptation-only) vs Cell 4b (GAN + adaptation) ===")
for model_label, results_4c, results_4b in [
    ("LSTM-XGB", lstm_xgb_4c_agg["raw_results"], lstm_xgb_4b_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4c_agg["raw_results"], cnn_lstm_4b_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4b, results_4c, metric="accuracy")
    sig = "SIGNIFICANT -- GAN's generalization step earns its cost" if diff_stats["p_value"] < 0.05 \
        else "NOT significant -- adaptation alone (4c) is sufficient; GAN adds no measurable benefit"
    print(f"  {model_label}: mean diff (4b - 4c) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")

print("\n=== Cell 2 (Stage 2, no GAN) vs Cell 4f (full-spectrum pool, unconditional GAN) ===")
for model_label, key, results_4f in [
    ("LSTM-XGB", "lstm_xgb", lstm_xgb_4f_agg["raw_results"]),
    ("CNN-LSTM-v2", "cnn_lstm_v2", cnn_lstm_4f_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4f, stage2[key]["raw_results"], metric="accuracy")
    print(f"  {model_label}: mean diff (4f - Stage2) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== THE FRAMEWORK: Cell 2 (Stage 2, no GAN) vs Cell 4g (stratified lifetime pool) ===")
for model_label, key, results_4g in [
    ("LSTM-XGB", "lstm_xgb", lstm_xgb_4g_agg["raw_results"]),
    ("CNN-LSTM-v2", "cnn_lstm_v2", cnn_lstm_4g_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4g, stage2[key]["raw_results"], metric="accuracy")
    print(f"  {model_label}: mean diff (4g - Stage2) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}")

print("\n=== Does stratification help? Cell 4f (uniform full-spectrum) vs Cell 4g (stratified + pristine tier) ===")
for model_label, results_4f, results_4g in [
    ("LSTM-XGB", lstm_xgb_4f_agg["raw_results"], lstm_xgb_4g_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4f_agg["raw_results"], cnn_lstm_4g_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4g, results_4f, metric="accuracy")
    sig = "SIGNIFICANT -- stratification + pristine tier adds real value over uniform sampling" if diff_stats["p_value"] < 0.05 \
        else "NOT significant -- uniform full-spectrum sampling was already sufficient"
    print(f"  {model_label}: mean diff (4g - 4f) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")

print("\n=== Is the GAN needed? Cell 4c (deterministic, no GAN) vs Cell 4g (stratified GAN framework) ===")
for model_label, results_4c, results_4g in [
    ("LSTM-XGB", lstm_xgb_4c_agg["raw_results"], lstm_xgb_4g_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4c_agg["raw_results"], cnn_lstm_4g_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4g, results_4c, metric="accuracy")
    sig = "SIGNIFICANT -- the stratified GAN framework beats the deterministic baseline" if diff_stats["p_value"] < 0.05 \
        else "NOT significant -- deterministic 4c matches the full GAN framework"
    print(f"  {model_label}: mean diff (4g - 4c) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")

print("\n=== THE CONDITIONING QUESTION: Cell 4f (full-spectrum, unconditional) vs Cell 4c (targeted projection, no GAN) ===")
for model_label, results_4f, results_4c in [
    ("LSTM-XGB", lstm_xgb_4f_agg["raw_results"], lstm_xgb_4c_agg["raw_results"]),
    ("CNN-LSTM-v2", cnn_lstm_4f_agg["raw_results"], cnn_lstm_4c_agg["raw_results"]),
]:
    diff_df, diff_stats = paired_diff(results_4c, results_4f, metric="accuracy")
    sig = "4c beats 4f -- targeting the test region beats blended full-spectrum exposure" if diff_stats["p_value"] < 0.05 and diff_stats["mean_diff"] > 0 \
        else ("4f beats 4c -- full-spectrum exposure alone was enough, unexpectedly" if diff_stats["p_value"] < 0.05 else "NOT significant")
    print(f"  {model_label}: mean diff (4c - 4f) = {diff_stats['mean_diff']:+.4f}  "
          f"t={diff_stats['t_stat']:.3f}  p={diff_stats['p_value']:.4f}   [{sig}]")



=== Cell 3 (Stage 2, no GAN) vs Cell 4a (GAN, vanilla) ===
  LSTM-XGB: mean diff = -0.0036  t=-0.126  p=0.9059
  CNN-LSTM-v2: mean diff = +0.0006  t=0.732  p=0.5049

=== Cell 2 (Stage 3, GAN no-drift) vs Cell 4a (GAN, drift) ===
  LSTM-XGB: mean diff = -0.1748  t=-4.786  p=0.0087
  CNN-LSTM-v2: mean diff = -0.1251  t=-612.781  p=0.0000

=== Cell 4a (vanilla) vs Cell 4b (domain-adaptation) ===
  LSTM-XGB: mean diff (4b - 4a) = +0.1730  t=6.266  p=0.0033
  CNN-LSTM-v2: mean diff (4b - 4a) = +0.1016  t=9.960  p=0.0006

=== Cell 2 (Stage 2, no GAN) vs Cell 4c (adaptation-only, no GAN) ===
  LSTM-XGB: mean diff (4c - Stage2) = +0.1458  t=4.612  p=0.0099
  CNN-LSTM-v2: mean diff (4c - Stage2) = +0.1193  t=26.287  p=0.0000

=== Cell 2 (Stage 2, no GAN) vs Cell 4d (row-count control, no shift) ===
  LSTM-XGB: mean diff (4d - Stage2) = -0.0788  t=-2.601  p=0.0600
  CNN-LSTM-v2: mean diff (4d - Stage2) = +0.0007  t=0.981  p=0.3821

=== THE SHIFT QUESTION: Cell 4d (oversample only) vs Cell 4c (o

## Deferred: Drift-Severity Sub-Matrix (Roadmap §5.2)

The roadmap's full design additionally sweeps TEST severity (no / mild /
matched / severe) specifically for Cell 4, to get a generalization response
curve rather than a single matched-severity point estimate. Not built out
in this notebook (roadmap explicitly deferred "full experimental
parameters... run logistics" pending Phase 4's injection function being
finalized, which it now is) — stubbed here as a concrete next step:

```python
def evaluate_at_severity(dct_builder, splits_variant, severity_multiplier, ...):
    """Re-run inject_drift's amplitude calibration at a different
    TARGET_SEVERITY_X (e.g. 0 for 'no drift', 2.0 for 'mild', 4.0 for
    'matched' -- the value used to train on -- and 6.0 for 'severe'),
    re-scale, and evaluate the ALREADY-TRAINED Stage 4 model against each
    resulting test set. Requires separating 'train severity' (fixed at
    injection time, baked into the trained model) from 'test severity'
    (swept post-hoc against a frozen model) -- i.e. re-inject drift into
    just the TEST partition at each severity level, holding the trained
    model and training data fixed.
    """
```

This is the natural next addition once 4a/4b's matched-severity results are
in and it's clear whether the domain-adaptation variant is worth the extra
complexity of a full severity sweep.


## Next Steps

- Run this notebook end-to-end. The five paired comparisons above are the
  headline results for the paper's Section on Stage 4.
- **The 4c-vs-4b comparison determines what this paper's method actually
  is.** If 4b beats 4c significantly, the contribution is a genuine hybrid
  — a generalization operator (WGAN-GP) combined with an adaptation
  operator (`project_to_tau`) — and it's worth framing the method as a
  general, decomposable G×A framework (either component swappable: a VAE
  or diffusion model for G, an estimated-drift version for A) rather than
  one fixed pipeline. If 4c matches or beats 4b, report the simpler,
  cheaper, GAN-free method instead — that's a stronger result, not a
  weaker one.
- If 4b's improvement over 4a is within the ~4.7-pt LSTM-XGB seed-noise
  band established in Stage 2, report it as "no significant improvement
  from drift-aware projection" rather than a positive result — the paired
  t-test above is exactly for catching that.
- Track F3 specifically in the per-class table across all three variants:
  Stage 2 showed it collapsing to ~1-3% regardless of model architecture.
  Whether 4c alone recovers it as well as 4b does is the most direct
  evidence for or against the hybrid claim.
